# Smart Retinal Camera AI — retrospective publication experiments

Run with an authorised APTOS 2019 Kaggle account. This notebook clones public analysis code but reads the original checkpoint and split manifests from your private Drive release. It does not alter the historical Colab notebook or checkpoint. Keep all per-image outputs private. No clinical validation is claimed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
release = Path('/content/drive/MyDrive/Smart-Retinal-Camera-AI/research_release_20261121T123249Z')
assert (release/'models/resnet18_epoch1.pth').is_file(), 'Check the private Drive release path'
assert all((release/'splits'/f'{name}.csv').is_file() for name in ('train','val','test'))


## Install code and download the authorised dataset

Paste the Kaggle API token at the hidden prompt. It is written only to the temporary Colab runtime with owner-only permissions. It is never printed or copied to Drive. The competition may require acceptance of its terms in your Kaggle account. If the download fails, stop and resolve access; do not substitute another dataset.


In [ ]:
!git clone --depth 1 https://github.com/Faisalhassan903/Smart-Retinal-Camera-AI.git /content/Smart-Retinal-Camera-AI
%cd /content/Smart-Retinal-Camera-AI
!python -m pip install -q kaggle six opencv-python-headless scikit-learn pandas matplotlib
from getpass import getpass
from pathlib import Path
token = getpass('Kaggle API token: ')
kaggle_dir = Path.home()/'.kaggle'
kaggle_dir.mkdir(mode=0o700, exist_ok=True)
token_file = kaggle_dir/'access_token'
token_file.write_text(token.strip())
token_file.chmod(0o600)
del token


In [ ]:
!mkdir -p /content/aptos
!kaggle competitions download -c aptos2019-blindness-detection -p /content/aptos
import zipfile
archives = list(Path('/content/aptos').glob('*.zip'))
for archive in archives:
    with zipfile.ZipFile(archive) as z:
        z.extractall('/content/aptos')
for archive in Path('/content/aptos').glob('*.zip'):
    if archive not in archives:
        with zipfile.ZipFile(archive) as z:
            z.extractall('/content/aptos')
token_file.unlink(missing_ok=True)
images = Path('/content/aptos/train_images')
assert images.is_dir(), 'Expected labelled APTOS train_images directory'
assert (Path('/content/aptos')/'train.csv').is_file()
print('Available PNG files:', len(list(images.glob('*.png'))))


## Preserve the original test partition and run experiments 1–5

The script verifies manifest IDs and labels, screens exact and perceptual duplicates across partitions, reproduces the original test evaluation, runs 2,000 stratified image-level bootstrap resamples, computes disease-grade results, tests fixed synthetic degradations, and measures fixed review-rule coverage. A perceptual-hash match is a candidate for manual inspection, not proof of the same patient. The test partition is never changed or used to tune thresholds.


In [ ]:
import shutil
project = Path('/content/Smart-Retinal-Camera-AI')
(project/'splits').mkdir(exist_ok=True)
(project/'models').mkdir(exist_ok=True)
for name in ('train','val','test'):
    shutil.copy2(release/'splits'/f'{name}.csv', project/'splits'/f'{name}.csv')
shutil.copy2(release/'models/resnet18_epoch1.pth', project/'models/resnet18_epoch1.pth')
assert (release/'results/test_predictions_private.csv').is_file()
(project/'private_archive').mkdir(exist_ok=True)
shutil.copy2(release/'results/test_predictions_private.csv', project/'private_archive/test_predictions_private.csv')
private_output = release/'publication_runs'
private_output.mkdir(exist_ok=True)


In [ ]:
!python -m src.publication_experiments --splits /content/Smart-Retinal-Camera-AI/splits --labels /content/aptos/train.csv --images /content/aptos/train_images --checkpoint /content/Smart-Retinal-Camera-AI/models/resnet18_epoch1.pth --saved-predictions /content/Smart-Retinal-Camera-AI/private_archive/test_predictions_private.csv --output "{private_output}" --bootstrap 2000


## Review the generated evidence

Open the newest `publication_runs/<timestamp>/status.json`. `historical_metrics_all_match` must be true before claiming numerical reproduction. Inspect `exact_cross_partition.csv` and `perceptual_candidates.csv` manually. The private `paired_predictions_PRIVATE.csv` contains image IDs, labels and scores. Do not publish it or the restricted dataset. Copy only aggregate, checked tables and figures into a manuscript. Patient-level separation remains unresolved without patient identifiers.


In [ ]:
import json
runs = sorted(p for p in private_output.iterdir() if p.is_dir())
latest = runs[-1]
print(latest)
print(json.dumps(json.loads((latest/'status.json').read_text()), indent=2))


## Optional: three seeded frozen-backbone training runs

Run only if GPU time permits. This is a new experiment. It never replaces the historical checkpoint, uses the preserved training and validation manifests, and writes each seed to a separate private Drive directory. Inspect `history.csv` and `config.json` for validation metrics and elapsed seconds. Select a model using validation data before any test evaluation.


In [ ]:
RUN_THREE_SEEDS = False
if RUN_THREE_SEEDS:
    import subprocess
    for seed in (11, 23, 37):
        command = ['python', '-m', 'src.train', '--csv', '/content/aptos/train.csv',
                   '--images', '/content/aptos/train_images', '--splits', str(project/'splits'),
                   '--output', str(release/'training_runs'/f'seed_{seed}'), '--epochs', '5',
                   '--batch-size', '32', '--lr', '0.001', '--seed', str(seed)]
        subprocess.run(command, cwd=project, check=True)
